# Strategy Workbench Colab Runner

이 노트북은 여러 전략을 한 번에 비교하고, 기간/이평선/분할매도/트레일링 스탑 같은 조건을 바꿔가며 백테스트할 수 있도록 만든 실행기입니다.

사용 방법:

1. 아래 `Global Settings`, `Strategy 1`, `Strategy 2`, `Strategy 3` 셀에서 값만 바꿉니다.
2. 셀을 위에서부터 순서대로 실행합니다.
3. `metrics.csv` 표와 비교 차트를 확인합니다.
4. 마지막 셀에서 결과 파일을 zip으로 내려받을 수 있습니다.

지금 버전은 `config 딕셔너리`를 직접 쓰지 않도록, 최대한 드롭다운과 숫자 입력 중심으로 바꿔둔 버전입니다.


In [ ]:
REPO_URL = "https://github.com/snowballTQ/multi-strategy-fucker.git"
BRANCH = "main"


In [ ]:
from pathlib import Path
import shutil
import subprocess

repo_dir = Path('/content/strategy_workbench_repo')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    'git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(repo_dir)
], check=True)

print(f'Repo directory: {repo_dir}')


## 입력 규칙 안내

이 Colab은 최대한 `폼 입력`만으로 전략을 바꿔볼 수 있게 만든 버전입니다.

큰 원칙은 이렇게 보시면 됩니다.

- 날짜는 달력에서 고르면 됩니다.
- 숫자로 바꿔야 하는 값은 직접 입력하면 됩니다.
- 글자로 고르는 값은 드롭다운으로 만들었습니다.
- `confirm_days`는 이번 Colab 폼에서는 뺐고, 기본적으로 1일 확인 기준으로 처리합니다.

입력 셀은 크게 네 덩어리입니다.

1. `Global Settings`
- 전체 백테스트 공통 설정입니다.
- `START_DATE`, `END_DATE`: 전체 비교 구간입니다.
- `COMMISSION_RATE`: 편도 수수료입니다. 예: `0.001` = 0.10%
- `TAX_RATE`: 세율입니다. 예: `0.22` = 22%
- `EXPENSE_RATIO`: 연 운용보수 가정값입니다. 예: `0.0095` = 0.95%
- `BORROW_SPREAD`: 차입 스프레드 가정값입니다. 예: `1.0`
- `OUTPUT_NAME`: 결과 폴더 이름입니다. 비교 실험마다 이름을 다르게 두면 편합니다.

2. `Strategy 1`, `Strategy 2`, `Strategy 3`
- 전략 슬롯입니다.
- `ENABLE_STRATEGY_X = True`면 그 전략을 실행합니다.
- `False`로 두면 그 슬롯은 완전히 무시됩니다.
- 즉 전략 1개만 돌리고 싶으면 1번만 켜고, 3개 비교하고 싶으면 3개를 모두 켜면 됩니다.

3. 전략 이름과 기본 자산 설정
- `STRATEGY_NAME_X`: 결과표와 차트에 보일 이름입니다.
- `BASE_SERIES_X`: 기준 시계열입니다.
  - `ndx`: 나스닥100
  - `spx`: S&P500
  - `composite_splice`: 이어붙인 장기 시계열
- `LEVERAGE_X`: 레버리지 배수입니다. 예: `1.0`, `2.0`, `3.0`
- `INCLUDE_FINANCING_COST_X`: 비용 반영 여부입니다. 보통 `True`를 권장합니다.
- `EXIT_DESTINATION_X`: 전량 청산 시 매도 대금을 어디로 보낼지 정합니다.

4. 규칙 설정
- 각 전략은 `ENTRY`와 `EXIT`를 따로 설정합니다.
- `COMBINE`은 규칙 2개를 어떻게 묶을지 정합니다.
  - `single`: RULE1만 사용
  - `all_of`: RULE1과 RULE2를 동시에 만족해야 함
  - `any_of`: RULE1 또는 RULE2 중 하나만 만족해도 됨
- `RULE1_TYPE`, `RULE2_TYPE`은 규칙 종류입니다.
- `RULE1_SOURCE`, `RULE2_SOURCE`는 어떤 시계열을 기준으로 규칙을 계산할지 정합니다.
  - `traded`: 실제 매매 대상 시계열
  - `base`: 그 전략의 원본 기준 시계열
  - `ndx`, `spx`, `composite_splice`: 특정 지수를 직접 기준으로 삼음

rule type별 window 입력법은 아래처럼 보시면 됩니다.

- `price_above_sma`, `price_below_sma`
  - `WINDOW1`만 씁니다.
  - 예: `WINDOW1 = 200`
  - 의미: 가격이 200일선 위/아래인지 판단

- `fast_above_slow`, `fast_below_slow`
  - `WINDOW1 = 빠른선`, `WINDOW2 = 느린선`
  - 예: `50`, `200`
  - 의미: 50일선이 200일선 위/아래인지 판단

- `sma_chain_above`, `sma_chain_below`
  - `WINDOW1 / WINDOW2 / WINDOW3`를 순서대로 씁니다.
  - 예: `3`, `161`, `185`
  - 의미: `sma(3) > sma(161) > sma(185)` 또는 반대 구조인지 판단
  - 2개만 쓰고 싶으면 2개만 넣어도 되지만, 이 Colab은 3칸까지 열어둔 상태입니다.

- `always_true`
  - window를 안 씁니다.
  - 규칙을 항상 참으로 두고 싶을 때 씁니다.

- `none`
  - RULE2를 끄고 싶을 때 씁니다.
  - `RULE2_TYPE = none`으로 두고 남는 window 값은 `0`으로 두면 됩니다.

분할매도 입력법은 이렇게 보시면 됩니다.

- `TP1_ENABLE / TP2_ENABLE / TP3_ENABLE`
  - 각 익절 구간 사용 여부입니다.
- `TP1_TRIGGER`
  - 진입가 대비 몇 배에서 익절할지 정합니다.
  - 예: `1.15` = +15%
- `TP1_SELL_FRACTION`
  - 현재 보유 물량 중 몇 %를 팔지 정합니다.
  - 예: `0.50` = 50%
- `TP1_DESTINATION`
  - 그 매도 대금을 어디로 보낼지 정합니다.

트레일링 스탑 입력법은 이렇게 보시면 됩니다.

- `ENABLE_TRAILING_X`
  - 트레일링 스탑 사용 여부입니다.
- `TRAILING_AFTER_FIRST_TP_X`
  - 첫 익절 이후에만 활성화할지 여부입니다.
- `TRAILING_DRAWDOWN_X`
  - 고점 대비 몇 % 하락하면 전량 청산할지 정합니다.
  - 예: `0.15` = 고점 대비 15% 하락 시 청산
- `TRAILING_DESTINATION_X`
  - 트레일링 스탑으로 매도한 대금을 어디로 보낼지 정합니다.

바로 써먹을 수 있는 예시는 아래처럼 생각하시면 됩니다.

- 예시 1: 200일선 타이밍 3배
  - ENTRY_RULE1_TYPE = `price_above_sma`
  - ENTRY_RULE1_WINDOW1 = `200`
  - EXIT_RULE1_TYPE = `price_below_sma`
  - EXIT_RULE1_WINDOW1 = `200`
  - LEVERAGE = `3.0`

- 예시 2: 3-161-185 조합 + 부분익절
  - ENTRY_COMBINE = `all_of`
  - ENTRY_RULE1_TYPE = `price_above_sma`, WINDOW1 = `200`
  - ENTRY_RULE2_TYPE = `sma_chain_above`, WINDOW1/WINDOW2/WINDOW3 = `3`, `161`, `185`
  - EXIT_COMBINE = `any_of`
  - EXIT_RULE1_TYPE = `price_below_sma`, WINDOW1 = `200`
  - EXIT_RULE2_TYPE = `sma_chain_below`, WINDOW1/WINDOW2/WINDOW3 = `3`, `161`, `185`
  - TP1_TRIGGER = `1.15`, TP1_SELL_FRACTION = `0.50`
  - TP2_TRIGGER = `1.68`, TP2_SELL_FRACTION = `0.35`

- 예시 3: 듀얼 이평선 교차 전략
  - ENTRY_RULE1_TYPE = `fast_above_slow`, WINDOW1 = `50`, WINDOW2 = `200`
  - EXIT_RULE1_TYPE = `fast_below_slow`, WINDOW1 = `50`, WINDOW2 = `200`

자주 쓰이는 숫자 예시는 이 정도입니다.

- 이평선: `3`, `5`, `20`, `50`, `60`, `120`, `161`, `185`, `200`
- 레버리지: `1.0`, `2.0`, `3.0`
- 익절 배수: `1.10`, `1.15`, `1.25`, `1.50`, `1.68`, `2.00`
- 트레일링 스탑: `0.10`, `0.15`, `0.20`

처음 쓰는 분께 추천하는 순서는 이렇습니다.

1. 먼저 기본값 그대로 실행합니다.
2. 그다음 전략 3을 켜서 비교 전략을 하나 더 추가합니다.
3. 그다음 LEVERAGE, WINDOW 값, TP 값만 조금씩 바꿔봅니다.
4. 마지막으로 EXIT_DESTINATION과 TRAILING을 켜서 차이를 봅니다.

실수 방지 팁:

- RULE2를 안 쓸 거면 `RULE2_TYPE = none`으로 두세요.
- `fast_above_slow`, `fast_below_slow`를 쓸 때는 보통 `WINDOW1 < WINDOW2`가 자연스럽습니다.
- `sma_chain_above`를 쓸 때는 짧은 이평선부터 긴 이평선 순으로 넣는 것이 보통 읽기 쉽습니다.
- 분할매도를 켰는데 목적지를 바꾸지 않으면 기본적으로 `cash`로 두는 것이 가장 단순합니다.
- 처음에는 전략 2개만 켜고 돌려본 뒤, 그다음 3개 비교로 늘리는 것을 추천합니다.


In [ ]:
# @title Global Settings
START_DATE = "1985-10-01" # @param {type:"date"}
END_DATE = "2026-03-09" # @param {type:"date"}
COMMISSION_RATE = 0.001 # @param {type:"number"}
TAX_RATE = 0.22 # @param {type:"number"}
EXPENSE_RATIO = 0.0095 # @param {type:"number"}
BORROW_SPREAD = 1.0 # @param {type:"number"}
OUTPUT_NAME = "strategy_workbench_demo" # @param {type:"string"}


In [ ]:
# @title Strategy 1
ENABLE_STRATEGY_1 = True # @param {type:"boolean"}
STRATEGY_NAME_1 = "Price vs 200 SMA | 3x" # @param {type:"string"}
BASE_SERIES_1 = "ndx" # @param ["ndx", "spx", "composite_splice"]
LEVERAGE_1 = 3.0 # @param {type:"number"}
INCLUDE_FINANCING_COST_1 = True # @param {type:"boolean"}
EXIT_DESTINATION_1 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
ENTRY_COMBINE_1 = "single" # @param ["single", "all_of", "any_of"]
ENTRY_RULE1_TYPE_1 = "price_above_sma" # @param ["price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
ENTRY_RULE1_SOURCE_1 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
ENTRY_RULE1_WINDOW1_1 = 200 # @param {type:"integer"}
ENTRY_RULE1_WINDOW2_1 = 0 # @param {type:"integer"}
ENTRY_RULE1_WINDOW3_1 = 0 # @param {type:"integer"}
ENTRY_RULE2_TYPE_1 = "none" # @param ["none", "price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
ENTRY_RULE2_SOURCE_1 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
ENTRY_RULE2_WINDOW1_1 = 0 # @param {type:"integer"}
ENTRY_RULE2_WINDOW2_1 = 0 # @param {type:"integer"}
ENTRY_RULE2_WINDOW3_1 = 0 # @param {type:"integer"}
EXIT_COMBINE_1 = "single" # @param ["single", "all_of", "any_of"]
EXIT_RULE1_TYPE_1 = "price_below_sma" # @param ["price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
EXIT_RULE1_SOURCE_1 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
EXIT_RULE1_WINDOW1_1 = 200 # @param {type:"integer"}
EXIT_RULE1_WINDOW2_1 = 0 # @param {type:"integer"}
EXIT_RULE1_WINDOW3_1 = 0 # @param {type:"integer"}
EXIT_RULE2_TYPE_1 = "none" # @param ["none", "price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
EXIT_RULE2_SOURCE_1 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
EXIT_RULE2_WINDOW1_1 = 0 # @param {type:"integer"}
EXIT_RULE2_WINDOW2_1 = 0 # @param {type:"integer"}
EXIT_RULE2_WINDOW3_1 = 0 # @param {type:"integer"}
TP1_ENABLE_1 = False # @param {type:"boolean"}
TP1_TRIGGER_1 = 1.15 # @param {type:"number"}
TP1_SELL_FRACTION_1 = 0.5 # @param {type:"number"}
TP1_DESTINATION_1 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
TP2_ENABLE_1 = False # @param {type:"boolean"}
TP2_TRIGGER_1 = 1.68 # @param {type:"number"}
TP2_SELL_FRACTION_1 = 0.35 # @param {type:"number"}
TP2_DESTINATION_1 = "spy" # @param ["cash", "sgov", "spy", "qqq", "base"]
TP3_ENABLE_1 = False # @param {type:"boolean"}
TP3_TRIGGER_1 = 2.0 # @param {type:"number"}
TP3_SELL_FRACTION_1 = 1.0 # @param {type:"number"}
TP3_DESTINATION_1 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
ENABLE_TRAILING_1 = False # @param {type:"boolean"}
TRAILING_AFTER_FIRST_TP_1 = True # @param {type:"boolean"}
TRAILING_DRAWDOWN_1 = 0.15 # @param {type:"number"}
TRAILING_DESTINATION_1 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]


In [ ]:
# @title Strategy 2
ENABLE_STRATEGY_2 = True # @param {type:"boolean"}
STRATEGY_NAME_2 = "3-161-185 + Partial Exits | 3x" # @param {type:"string"}
BASE_SERIES_2 = "ndx" # @param ["ndx", "spx", "composite_splice"]
LEVERAGE_2 = 3.0 # @param {type:"number"}
INCLUDE_FINANCING_COST_2 = True # @param {type:"boolean"}
EXIT_DESTINATION_2 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
ENTRY_COMBINE_2 = "all_of" # @param ["single", "all_of", "any_of"]
ENTRY_RULE1_TYPE_2 = "price_above_sma" # @param ["price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
ENTRY_RULE1_SOURCE_2 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
ENTRY_RULE1_WINDOW1_2 = 200 # @param {type:"integer"}
ENTRY_RULE1_WINDOW2_2 = 0 # @param {type:"integer"}
ENTRY_RULE1_WINDOW3_2 = 0 # @param {type:"integer"}
ENTRY_RULE2_TYPE_2 = "sma_chain_above" # @param ["none", "price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
ENTRY_RULE2_SOURCE_2 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
ENTRY_RULE2_WINDOW1_2 = 3 # @param {type:"integer"}
ENTRY_RULE2_WINDOW2_2 = 161 # @param {type:"integer"}
ENTRY_RULE2_WINDOW3_2 = 185 # @param {type:"integer"}
EXIT_COMBINE_2 = "any_of" # @param ["single", "all_of", "any_of"]
EXIT_RULE1_TYPE_2 = "price_below_sma" # @param ["price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
EXIT_RULE1_SOURCE_2 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
EXIT_RULE1_WINDOW1_2 = 200 # @param {type:"integer"}
EXIT_RULE1_WINDOW2_2 = 0 # @param {type:"integer"}
EXIT_RULE1_WINDOW3_2 = 0 # @param {type:"integer"}
EXIT_RULE2_TYPE_2 = "sma_chain_below" # @param ["none", "price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
EXIT_RULE2_SOURCE_2 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
EXIT_RULE2_WINDOW1_2 = 3 # @param {type:"integer"}
EXIT_RULE2_WINDOW2_2 = 161 # @param {type:"integer"}
EXIT_RULE2_WINDOW3_2 = 185 # @param {type:"integer"}
TP1_ENABLE_2 = True # @param {type:"boolean"}
TP1_TRIGGER_2 = 1.15 # @param {type:"number"}
TP1_SELL_FRACTION_2 = 0.5 # @param {type:"number"}
TP1_DESTINATION_2 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
TP2_ENABLE_2 = True # @param {type:"boolean"}
TP2_TRIGGER_2 = 1.68 # @param {type:"number"}
TP2_SELL_FRACTION_2 = 0.35 # @param {type:"number"}
TP2_DESTINATION_2 = "spy" # @param ["cash", "sgov", "spy", "qqq", "base"]
TP3_ENABLE_2 = False # @param {type:"boolean"}
TP3_TRIGGER_2 = 2.0 # @param {type:"number"}
TP3_SELL_FRACTION_2 = 1.0 # @param {type:"number"}
TP3_DESTINATION_2 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
ENABLE_TRAILING_2 = True # @param {type:"boolean"}
TRAILING_AFTER_FIRST_TP_2 = True # @param {type:"boolean"}
TRAILING_DRAWDOWN_2 = 0.15 # @param {type:"number"}
TRAILING_DESTINATION_2 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]


In [ ]:
# @title Strategy 3
ENABLE_STRATEGY_3 = False # @param {type:"boolean"}
STRATEGY_NAME_3 = "Dual SMA Crossover | 2x" # @param {type:"string"}
BASE_SERIES_3 = "ndx" # @param ["ndx", "spx", "composite_splice"]
LEVERAGE_3 = 2.0 # @param {type:"number"}
INCLUDE_FINANCING_COST_3 = True # @param {type:"boolean"}
EXIT_DESTINATION_3 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
ENTRY_COMBINE_3 = "single" # @param ["single", "all_of", "any_of"]
ENTRY_RULE1_TYPE_3 = "fast_above_slow" # @param ["price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
ENTRY_RULE1_SOURCE_3 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
ENTRY_RULE1_WINDOW1_3 = 50 # @param {type:"integer"}
ENTRY_RULE1_WINDOW2_3 = 200 # @param {type:"integer"}
ENTRY_RULE1_WINDOW3_3 = 0 # @param {type:"integer"}
ENTRY_RULE2_TYPE_3 = "none" # @param ["none", "price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
ENTRY_RULE2_SOURCE_3 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
ENTRY_RULE2_WINDOW1_3 = 0 # @param {type:"integer"}
ENTRY_RULE2_WINDOW2_3 = 0 # @param {type:"integer"}
ENTRY_RULE2_WINDOW3_3 = 0 # @param {type:"integer"}
EXIT_COMBINE_3 = "single" # @param ["single", "all_of", "any_of"]
EXIT_RULE1_TYPE_3 = "fast_below_slow" # @param ["price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
EXIT_RULE1_SOURCE_3 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
EXIT_RULE1_WINDOW1_3 = 50 # @param {type:"integer"}
EXIT_RULE1_WINDOW2_3 = 200 # @param {type:"integer"}
EXIT_RULE1_WINDOW3_3 = 0 # @param {type:"integer"}
EXIT_RULE2_TYPE_3 = "none" # @param ["none", "price_above_sma", "price_below_sma", "sma_chain_above", "sma_chain_below", "fast_above_slow", "fast_below_slow", "always_true"]
EXIT_RULE2_SOURCE_3 = "traded" # @param ["traded", "base", "ndx", "spx", "composite_splice"]
EXIT_RULE2_WINDOW1_3 = 0 # @param {type:"integer"}
EXIT_RULE2_WINDOW2_3 = 0 # @param {type:"integer"}
EXIT_RULE2_WINDOW3_3 = 0 # @param {type:"integer"}
TP1_ENABLE_3 = False # @param {type:"boolean"}
TP1_TRIGGER_3 = 1.15 # @param {type:"number"}
TP1_SELL_FRACTION_3 = 0.5 # @param {type:"number"}
TP1_DESTINATION_3 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
TP2_ENABLE_3 = False # @param {type:"boolean"}
TP2_TRIGGER_3 = 1.68 # @param {type:"number"}
TP2_SELL_FRACTION_3 = 0.35 # @param {type:"number"}
TP2_DESTINATION_3 = "spy" # @param ["cash", "sgov", "spy", "qqq", "base"]
TP3_ENABLE_3 = False # @param {type:"boolean"}
TP3_TRIGGER_3 = 2.0 # @param {type:"number"}
TP3_SELL_FRACTION_3 = 1.0 # @param {type:"number"}
TP3_DESTINATION_3 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]
ENABLE_TRAILING_3 = False # @param {type:"boolean"}
TRAILING_AFTER_FIRST_TP_3 = True # @param {type:"boolean"}
TRAILING_DRAWDOWN_3 = 0.15 # @param {type:"number"}
TRAILING_DESTINATION_3 = "cash" # @param ["cash", "sgov", "spy", "qqq", "base"]


In [ ]:
import json

def build_atomic_rule(rule_type, source, window1, window2, window3):
    if rule_type == 'none':
        return None
    if rule_type == 'always_true':
        return {'type': 'always_true'}
    if rule_type in {'price_above_sma', 'price_below_sma'}:
        return {
            'type': rule_type,
            'source': source,
            'window': int(window1),
        }
    if rule_type in {'fast_above_slow', 'fast_below_slow'}:
        return {
            'type': rule_type,
            'source': source,
            'fast_window': int(window1),
            'slow_window': int(window2),
        }
    if rule_type in {'sma_chain_above', 'sma_chain_below'}:
        windows = [int(value) for value in (window1, window2, window3) if int(value) > 0]
        if len(windows) < 2:
            raise ValueError(f'{rule_type} needs at least two window values.')
        return {
            'type': rule_type,
            'source': source,
            'windows': windows,
        }
    raise ValueError(f'Unsupported rule type: {rule_type}')

def build_rule_group(combine_mode, rule1, rule2):
    valid_rules = [rule for rule in (rule1, rule2) if rule is not None]
    if not valid_rules:
        return {'type': 'always_true'}
    if combine_mode == 'single' or len(valid_rules) == 1:
        return valid_rules[0]
    return {
        'type': combine_mode,
        'rules': valid_rules,
    }

def build_take_profit_ladder(tp_rows):
    ladder = []
    for enabled, trigger, sell_fraction, destination in tp_rows:
        if enabled:
            ladder.append({
                'trigger_gain_multiple': float(trigger),
                'sell_fraction': float(sell_fraction),
                'destination': destination,
            })
    ladder.sort(key=lambda row: row['trigger_gain_multiple'])
    return ladder

def build_strategy(enabled, name, base_series, leverage, include_financing_cost, exit_destination,
                   entry_combine, entry_rule1_type, entry_rule1_source, entry_rule1_window1, entry_rule1_window2, entry_rule1_window3,
                   entry_rule2_type, entry_rule2_source, entry_rule2_window1, entry_rule2_window2, entry_rule2_window3,
                   exit_combine, exit_rule1_type, exit_rule1_source, exit_rule1_window1, exit_rule1_window2, exit_rule1_window3,
                   exit_rule2_type, exit_rule2_source, exit_rule2_window1, exit_rule2_window2, exit_rule2_window3,
                   tp1_enable, tp1_trigger, tp1_sell_fraction, tp1_destination,
                   tp2_enable, tp2_trigger, tp2_sell_fraction, tp2_destination,
                   tp3_enable, tp3_trigger, tp3_sell_fraction, tp3_destination,
                   enable_trailing, trailing_after_first_tp, trailing_drawdown, trailing_destination):
    if not enabled:
        return None

    entry_rule1 = build_atomic_rule(entry_rule1_type, entry_rule1_source, entry_rule1_window1, entry_rule1_window2, entry_rule1_window3)
    entry_rule2 = build_atomic_rule(entry_rule2_type, entry_rule2_source, entry_rule2_window1, entry_rule2_window2, entry_rule2_window3)
    exit_rule1 = build_atomic_rule(exit_rule1_type, exit_rule1_source, exit_rule1_window1, exit_rule1_window2, exit_rule1_window3)
    exit_rule2 = build_atomic_rule(exit_rule2_type, exit_rule2_source, exit_rule2_window1, exit_rule2_window2, exit_rule2_window3)

    return {
        'name': name,
        'base_series': base_series,
        'leverage': float(leverage),
        'include_financing_cost': bool(include_financing_cost),
        'exit_destination': exit_destination,
        'entry': build_rule_group(entry_combine, entry_rule1, entry_rule2),
        'exit': build_rule_group(exit_combine, exit_rule1, exit_rule2),
        'take_profit_ladder': build_take_profit_ladder([
            (tp1_enable, tp1_trigger, tp1_sell_fraction, tp1_destination),
            (tp2_enable, tp2_trigger, tp2_sell_fraction, tp2_destination),
            (tp3_enable, tp3_trigger, tp3_sell_fraction, tp3_destination),
        ]),
        'trailing_stop': {
            'enabled': bool(enable_trailing),
            'activate_after_first_take_profit': bool(trailing_after_first_tp),
            'drawdown_from_peak': float(trailing_drawdown),
            'destination': trailing_destination,
        },
    }

strategies = []
for strategy in [
    build_strategy(
        ENABLE_STRATEGY_1, STRATEGY_NAME_1, BASE_SERIES_1, LEVERAGE_1, INCLUDE_FINANCING_COST_1, EXIT_DESTINATION_1,
        ENTRY_COMBINE_1, ENTRY_RULE1_TYPE_1, ENTRY_RULE1_SOURCE_1, ENTRY_RULE1_WINDOW1_1, ENTRY_RULE1_WINDOW2_1, ENTRY_RULE1_WINDOW3_1,
        ENTRY_RULE2_TYPE_1, ENTRY_RULE2_SOURCE_1, ENTRY_RULE2_WINDOW1_1, ENTRY_RULE2_WINDOW2_1, ENTRY_RULE2_WINDOW3_1,
        EXIT_COMBINE_1, EXIT_RULE1_TYPE_1, EXIT_RULE1_SOURCE_1, EXIT_RULE1_WINDOW1_1, EXIT_RULE1_WINDOW2_1, EXIT_RULE1_WINDOW3_1,
        EXIT_RULE2_TYPE_1, EXIT_RULE2_SOURCE_1, EXIT_RULE2_WINDOW1_1, EXIT_RULE2_WINDOW2_1, EXIT_RULE2_WINDOW3_1,
        TP1_ENABLE_1, TP1_TRIGGER_1, TP1_SELL_FRACTION_1, TP1_DESTINATION_1,
        TP2_ENABLE_1, TP2_TRIGGER_1, TP2_SELL_FRACTION_1, TP2_DESTINATION_1,
        TP3_ENABLE_1, TP3_TRIGGER_1, TP3_SELL_FRACTION_1, TP3_DESTINATION_1,
        ENABLE_TRAILING_1, TRAILING_AFTER_FIRST_TP_1, TRAILING_DRAWDOWN_1, TRAILING_DESTINATION_1,
    ),
    build_strategy(
        ENABLE_STRATEGY_2, STRATEGY_NAME_2, BASE_SERIES_2, LEVERAGE_2, INCLUDE_FINANCING_COST_2, EXIT_DESTINATION_2,
        ENTRY_COMBINE_2, ENTRY_RULE1_TYPE_2, ENTRY_RULE1_SOURCE_2, ENTRY_RULE1_WINDOW1_2, ENTRY_RULE1_WINDOW2_2, ENTRY_RULE1_WINDOW3_2,
        ENTRY_RULE2_TYPE_2, ENTRY_RULE2_SOURCE_2, ENTRY_RULE2_WINDOW1_2, ENTRY_RULE2_WINDOW2_2, ENTRY_RULE2_WINDOW3_2,
        EXIT_COMBINE_2, EXIT_RULE1_TYPE_2, EXIT_RULE1_SOURCE_2, EXIT_RULE1_WINDOW1_2, EXIT_RULE1_WINDOW2_2, EXIT_RULE1_WINDOW3_2,
        EXIT_RULE2_TYPE_2, EXIT_RULE2_SOURCE_2, EXIT_RULE2_WINDOW1_2, EXIT_RULE2_WINDOW2_2, EXIT_RULE2_WINDOW3_2,
        TP1_ENABLE_2, TP1_TRIGGER_2, TP1_SELL_FRACTION_2, TP1_DESTINATION_2,
        TP2_ENABLE_2, TP2_TRIGGER_2, TP2_SELL_FRACTION_2, TP2_DESTINATION_2,
        TP3_ENABLE_2, TP3_TRIGGER_2, TP3_SELL_FRACTION_2, TP3_DESTINATION_2,
        ENABLE_TRAILING_2, TRAILING_AFTER_FIRST_TP_2, TRAILING_DRAWDOWN_2, TRAILING_DESTINATION_2,
    ),
    build_strategy(
        ENABLE_STRATEGY_3, STRATEGY_NAME_3, BASE_SERIES_3, LEVERAGE_3, INCLUDE_FINANCING_COST_3, EXIT_DESTINATION_3,
        ENTRY_COMBINE_3, ENTRY_RULE1_TYPE_3, ENTRY_RULE1_SOURCE_3, ENTRY_RULE1_WINDOW1_3, ENTRY_RULE1_WINDOW2_3, ENTRY_RULE1_WINDOW3_3,
        ENTRY_RULE2_TYPE_3, ENTRY_RULE2_SOURCE_3, ENTRY_RULE2_WINDOW1_3, ENTRY_RULE2_WINDOW2_3, ENTRY_RULE2_WINDOW3_3,
        EXIT_COMBINE_3, EXIT_RULE1_TYPE_3, EXIT_RULE1_SOURCE_3, EXIT_RULE1_WINDOW1_3, EXIT_RULE1_WINDOW2_3, EXIT_RULE1_WINDOW3_3,
        EXIT_RULE2_TYPE_3, EXIT_RULE2_SOURCE_3, EXIT_RULE2_WINDOW1_3, EXIT_RULE2_WINDOW2_3, EXIT_RULE2_WINDOW3_3,
        TP1_ENABLE_3, TP1_TRIGGER_3, TP1_SELL_FRACTION_3, TP1_DESTINATION_3,
        TP2_ENABLE_3, TP2_TRIGGER_3, TP2_SELL_FRACTION_3, TP2_DESTINATION_3,
        TP3_ENABLE_3, TP3_TRIGGER_3, TP3_SELL_FRACTION_3, TP3_DESTINATION_3,
        ENABLE_TRAILING_3, TRAILING_AFTER_FIRST_TP_3, TRAILING_DRAWDOWN_3, TRAILING_DESTINATION_3,
    ),
]:
    if strategy is not None:
        strategies.append(strategy)

CONFIG = {
    'start_date': START_DATE,
    'end_date': END_DATE,
    'commission_rate': float(COMMISSION_RATE),
    'tax_rate': float(TAX_RATE),
    'expense_ratio': float(EXPENSE_RATIO),
    'borrow_spread': float(BORROW_SPREAD),
    'strategies': strategies,
}

print('Current strategies:')
for strategy in CONFIG['strategies']:
    print('-', strategy['name'])

print('\nGenerated config preview:')
print(json.dumps(CONFIG, indent=2))


In [ ]:
import json
import subprocess
from pathlib import Path

config_path = Path('/content/workbench_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')

subprocess.run(
    ['python', 'run_workbench.py', '--config', str(config_path), '--output-name', OUTPUT_NAME],
    cwd=repo_dir,
    check=True,
)

print(f'Run complete: {OUTPUT_NAME}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

output_dir = Path('/root/strategy_workbench_outputs') / OUTPUT_NAME
metrics = pd.read_csv(output_dir / 'metrics.csv')
display(metrics)

curves = pd.read_csv(output_dir / 'equity_curves.csv')
pivot = curves.pivot(index='date', columns='strategy', values='equity')
pivot.plot(figsize=(12, 6), title='Strategy Comparison')
plt.ylabel('Equity Multiple')
plt.show()


In [ ]:
!cd /root && zip -r /content/strategy_workbench_outputs.zip strategy_workbench_outputs

from google.colab import files
files.download('/content/strategy_workbench_outputs.zip')
